## 1. Import the libraries:

In [3]:
os.chdir("../")
%pwd

'D:\\ML and AI Engineer\\Projects\\An-end-to-end-customer-churn-prediction-model'

In [37]:
import pandas as pd
import os
import numpy as np
import sys
from sklearn.impute import SimpleImputer
from src.customer_churn.entity.artifact_entity import DataValidationArtifacts, DataCleaningArtifacts
from src.customer_churn.config.configuration import DataCleaningConfig
from src.customer_churn.logging.logger import logging
from src.customer_churn.exception.exception import CustomerChurnException
from src.customer_churn.utils.main_utils.common import read_yaml_file
import joblib
import yaml

## 2. Reading the training data:

In [5]:
train = pd.read_parquet("Artifacts/data_validation/valid/train.parquet")
train.head(2)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,537472,22594,CHRISTMAS GINGHAM TREE,24,2010-12-07 11:11:00,0.85,12971,United Kingdom
1,532756,22984,"CARD, GINGHAM ROSE",12,2010-11-14 13:39:00,0.42,14769,United Kingdom


In [6]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 410693 entries, 0 to 410692
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      410693 non-null  object        
 1   StockCode    410693 non-null  object        
 2   Description  410693 non-null  object        
 3   Quantity     410693 non-null  int64         
 4   InvoiceDate  410693 non-null  datetime64[ns]
 5   Price        410693 non-null  float64       
 6   Customer ID  410693 non-null  int64         
 7   Country      410693 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(2), object(4)
memory usage: 25.1+ MB


## 3. Data Cleaning:

In [7]:
train.nunique()

Invoice        27324
StockCode       4076
Description     4529
Quantity         474
InvoiceDate    25420
Price            657
Customer ID     4749
Country           40
dtype: int64

In [8]:
train.shape

(410693, 8)

# Create a function here:

In [32]:
test = read_yaml_file("params.yaml")['data_cleaning']

In [33]:
test

{'imputation_strategy': 'median', 'capping_quantiles': 0.99}

In [31]:
test['data_cleaning']

{'imputation_strategy': 'median', 'capping_quantiles': 0.99}

In [55]:
class DataCleaning:
    def __init__(self, data_cleaning_config: DataCleaningConfig, data_validation_artifacts: DataValidationArtifacts):
        try:
            self.data_cleaning_config = data_cleaning_config
            self.data_validation_artifacts = data_validation_artifacts
            self.train_file_path = self.data_validation_artifacts.valid_train_file_path
            self.test_file_path = self.data_validation_artifacts.valid_test_file_path
            self.data_cleaning_params = read_yaml_file("params.yaml")['data_cleaning']
        except Exception as e:
            raise CustomerChurnException(e, sys)

    def handle_duplicates(self, df:pd.DataFrame, df_name:str) -> pd.DataFrame:
        try:
            duplicates = df.duplicated()
            if duplicates.any():
                num_of_duplicates = df[duplicates].shape[0]
                df.drop_duplicates(inplace=True)
                logging.info(f"Total number of duplicates in the given {df_name} dataframe: {num_of_duplicates:,} were dropped.")
            else:
                logging.info(f"No duplicates in {df_name} dataframe.")
            return df
        except Exception as e:
            raise CustomerChurnException(e, sys)

    def fit_price_imputer(self, train_df: pd.DataFrame) -> os.path:
        """
        This function fits and saves the imputer as .pkl file.
        """
        try:
            strategy = self.data_cleaning_params['imputation_strategy']
            imputer_path = self.data_cleaning_config.missing_data_imputer_file
            
            price_imputer = SimpleImputer(strategy=strategy)
            price_imputer.fit(train_df[['Price']])
            os.makedirs(os.path.dirname(imputer_path), exist_ok=True)            
            joblib.dump(price_imputer, imputer_path)
            logging.info(f"SimpleImputer used {strategy} as strategy and saved the imputer at: {imputer_path} path.")
            return imputer_path
        except Exception as e:
            raise CustomerChurnException(e, sys)

            
    def transform_price_imputer(self, df:pd.DataFrame, df_name:str) -> pd.DataFrame:
        try:
            imputer_path = self.data_cleaning_config.missing_data_imputer_file
            imputer = joblib.load(imputer_path)
            logging.info(f"Imputer loading Success from {imputer_path}")
            df['Price'] = imputer.transform(df[['Price']]).ravel()
            logging.info(f"Dataframe: {df_name} on column Price, imputation success!")
            return df
        except Exception as e:
            raise CustomerChurnException(e, sys)

    
    def report_missing_values(self, df:pd.DataFrame, df_name:str) -> bool:
        try:
            missing_report = False
            missing_vals = df.isna().sum()            
            if (missing_vals > 0).any():
                missing_report = True
                logging.info(f"{df_name} dataframe missing value report:\n{missing_vals}")
                return missing_report
            else:
                logging.info(f"{df_name} has no missing(nan) values.")
                return missing_report
        except Exception as e:
            raise CustomerChurnException(e, sys)
        

    def handle_rtrn_quantity_and_cncled_orders(self, df:pd.DataFrame, df_name:str) -> pd.DataFrame:
        """
        For returned orders flag 1 else 0
        For cancelled orders flag 1 else 0
        Creates a new column "Quantity_abs" with absolute values for Quantity column for feature selection in later stage.
        """
        try:
            df['orders_returned'] = [1 if q < 0  else 0 for q in df['Quantity']]
            df['orders_cancelled'] = [1 if invoice.startswith('C') else 0 for invoice in df['Invoice']]
            df['Quantity_abs'] = df['Quantity'].map(lambda q: np.abs(q))
            logging.info(f"Total number of returned orders in {df_name} dataframe: {df['orders_returned'].sum():,}")
            logging.info(f"Total number of cancelled orders in {df_name} dataframe: {df['orders_cancelled'].sum():,}")
            return df
        except Exception as e:
            raise CustomerChurnException(e, sys)

    def handle_outliers_w_clipping(self, df:pd.DataFrame, caps=None) -> pd.DataFrame:
        """
        Clips two columns: ['Quantity_abs', 'Price'] with 99th Quantile to test linear models.
        """
        
        try:
            # For training data:
            if caps is None:
                capping_quantile = self.data_cleaning_params['capping_quantiles']
                caps = {'Quantity_abs': df['Quantity_abs'].quantile(capping_quantile), 
                        'Price': df['Price'].quantile(capping_quantile)}
                
                # Save caps in the .yaml file to be used for testing file:
                caps_file_path_yaml = self.data_cleaning_config.caps_yaml_file
                with open(self.data_cleaning_config.caps_yaml_file, "w") as f:
                    yaml.dump(caps, f)
                logging.info(f"Clipping data from training file saved at: {caps_file_path_yaml}")

                # Clip the training data:
                for col, cap in caps.items():
                    df[f"{col}_clipped"] = df[col].clip(upper=cap)
                    logging.info(f"Column: {col} has been clipped by 99th quantile: {cap}")                
                return df, caps_file_path_yaml

            # For testing data:
            for col, cap in caps.items():
                df[f"{col}_clipped"] = df[col].clip(upper=cap)
                logging.info(f"Column: {col} has been clipped using {cap}")
            return df
        except Exception as e:
            raise CustomerChurnException(e, sys)

    
    def initiate_cleaning_training_data(self):
        """
        Cleans the training data:
        """
        imputer_path=None
        try:
            train_df = pd.read_parquet(self.train_file_path)
            logging.info("Reading training data success!")
            train_df_duplicates_removed = self.handle_duplicates(df=train_df, df_name='training')
            logging.info("Duplicates removed in the training data.")
            data_missing = self.report_missing_values(df=train_df_duplicates_removed, df_name='training')
            imputer_path = self.fit_price_imputer(train_df=train_df_duplicates_removed)
            if data_missing:
                missing_data_imputed_df = self.transform_price_imputer(df=train_df_duplicates_removed, df_name='training')
                clean_df = self.handle_rtrn_quantity_and_cncled_orders(df=missing_data_imputed_df, df_name='training')
            else:
                clean_df = self.handle_rtrn_quantity_and_cncled_orders(df=train_df_duplicates_removed, df_name='training')
                
            # Create a copy of the dataframe for linear and non-linear models:
            unclipped_df = clean_df.copy()
            
             # Save the uclipped training df file:
            unclipped_training_filepath = self.data_cleaning_config.unclipped_train_file_path
            os.makedirs(os.path.dirname(unclipped_training_filepath), exist_ok=True)         
            unclipped_df.to_parquet(unclipped_training_filepath, index=False)
            logging.info(f"Unclipped training data saved for tree models in: {unclipped_training_filepath}")

            # Clip the training data:
            clipped_training_filepath = self.data_cleaning_config.clipped_train_file_path
            os.makedirs(os.path.dirname(clipped_training_filepath), exist_ok=True)
            clipped_df, caps_file_path_yaml = self.handle_outliers_w_clipping(df=clean_df)
            clipped_df.to_parquet(clipped_training_filepath, index=False)
            logging.info(f"Clipped training data saved for linear models in: {clipped_training_filepath}")

            # Return the artifacts:
            data_cleaning_training_df_artifacts = DataCleaningArtifacts(
                clipped_train_file_path=clipped_training_filepath,
                clipped_testing_file_path=None,
                unclipped_training_file_path=unclipped_training_filepath,
                unclipped_testing_file_path=None,
                simple_imputer_file_path=imputer_path,
                caps_file_path_yaml = caps_file_path_yaml
            )           
        except Exception as e:
            raise CustomerChurnException(e, sys)
        
        
    def initiate_cleaning_test_data(self):
        """
        Cleans the testing data:
        """
       
        try:
            test_df = pd.read_parquet(self.test_file_path)
            logging.info("Reading testing data success!!")
            test_df_duplicates_removed = self.handle_duplicates(df=test_df, df_name='testing')
            logging.info("Duplicates removed in the testing data.")
            has_missing_price = self.report_missing_values(df=test_df_duplicates_removed, df_name='testing')
            if has_missing_price:
                missing_data_imputed_df = self.transform_price_imputer(df=test_df_duplicates_removed, df_name='testing')
                clean_df = self.handle_rtrn_quantity_and_cncled_orders(df=missing_data_imputed_df, df_name='testing')
            else:
                logging.info("Testing data has no missing values (NaN)")
                clean_df = self.handle_rtrn_quantity_and_cncled_orders(df=test_df_duplicates_removed)

            # Create a copy of the dataframe for linear and non-linear model:
            unclipped_df = clean_df.copy()

            # Save the unclipped testing dataframe:
            unclipped_testing_filepath = self.data_cleaning_config.unclipped_testing_file_path
            os.makedirs(os.path.dirname(unclipped_testing_filepath), exist_ok=True)
            unclipped_df.to_parquet(unclipped_testing_filepath, index=False)
            logging.info(f"Unclipped test file saved at: {unclipped_testing_filepath}")

            # Clip the testing file:
            clipped_testing_filepath = self.data_cleaning_config.clipped_test_file_path
            os.makedirs(os.path.dirname(clipped_testing_filepath), exist_ok=True)
            cap_file = read_yaml_file(self.data_cleaning_config.caps_yaml_file)
            clipped_df = self.handle_outliers_w_clipping(df=clean_df, caps=cap_file)
            clipped_df.to_parquet(clipped_testing_filepath, index=False)
            logging.info(f"Clipped testing data saved at: {clipped_testing_filepath}")

            data_cleaning_testing_df_artifacts = DataCleaningArtifacts(
                clipped_train_file_path=None,
                clipped_testing_file_path=clipped_testing_filepath,
                unclipped_training_file_path=None,
                unclipped_testing_file_path=unclipped_testing_filepath,
                simple_imputer_file_path=None,
                caps_file_path_yaml = None
            )            
        except Exception as e:
            raise CustomerChurnException(e, sys)
            
        
    

### 3.1 Check for duplicates:

In [11]:
duplicates = train.duplicated()
print(duplicates.any(), "\n")

True 



In [12]:
print(f"There is a total of: {train[duplicates].shape[0]:,} duplicate rows. Original data has: {train.shape[0]:,} rows.")
train.drop_duplicates(inplace=True)
print(f"Number of rows after dropping the duplicates: {train.shape[0]:,}")

There is a total of: 14,529 duplicate rows. Original data has: 410,693 rows.
Number of rows after dropping the duplicates: 396,164


### 3.2 Check for 0s:

In [13]:
(train==0.0).sum()

Invoice         0
StockCode       0
Description     0
Quantity        0
InvoiceDate     0
Price          30
Customer ID     0
Country         0
dtype: int64

#### Replace 0s with `np.nan` for correct statistical calculations:

In [14]:
train['Price'] = train['Price'].replace(0.0, np.nan)

In [15]:
(train==0).sum()

Invoice        0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
Price          0
Customer ID    0
Country        0
dtype: int64

In [17]:
null_vals = train.isna().sum()
null_vals

Invoice         0
StockCode       0
Description     0
Quantity        0
InvoiceDate     0
Price          30
Customer ID     0
Country         0
dtype: int64

In [24]:
if (null_vals > 0).any():
    print("Missing vals")
else: 
    print("No missing vals")

Missing vals


### Imputing the missing or `np.nan` values:

In [25]:
# Create a price imputer:
price_imputer = SimpleImputer(strategy='median')
price_imputer.fit(train[['Price']])

# Transform:
train['Price'] = price_imputer.transform(train[['Price']])

In [27]:
null_vals = train.isna().sum()
if (null_vals > 0).any():
    print("Missing vals")
else: 
    print("No missing vals")

No missing vals


## 3.2 Handling Returned Quantity & Cancelled Orders:

In [15]:
train.head(2)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,537472,22594,CHRISTMAS GINGHAM TREE,24,2010-12-07 11:11:00,0.85,12971,United Kingdom
1,532756,22984,"CARD, GINGHAM ROSE",12,2010-11-14 13:39:00,0.42,14769,United Kingdom


In [16]:
train['orders_returned'] = [1 if q < 0  else 0 for q in train['Quantity']]
print(f"A total of {train.value_counts('orders_returned').iloc[1]:,} orders were returned.")

A total of 9,588 orders were returned.


In [17]:
train['orders_cancelled'] = [1 if invoice.startswith('C') else 0 for invoice in train['Invoice']]
print(f"A total of {train.value_counts('orders_cancelled').iloc[1]:,} cancelled orders were found.")

A total of 9,588 cancelled orders were found.


#### Create a new column quantities as absolute values for the returned number of quantities:

In [44]:
train['Quantity_abs'] = train['Quantity'].map(lambda q: np.abs(q))

# Create the copy of the cleaned dataset:

In [48]:
train_copy_for_tree = train.copy()

In [49]:
train.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,orders_returned,orders_cancelled,Quantity_abs
0,537472,22594,CHRISTMAS GINGHAM TREE,24,2010-12-07 11:11:00,0.85,12971,United Kingdom,0,0,24
1,532756,22984,"CARD, GINGHAM ROSE",12,2010-11-14 13:39:00,0.42,14769,United Kingdom,0,0,12
2,527103,51014L,"FEATHER PEN,LIGHT PINK",24,2010-10-14 16:54:00,0.85,15549,United Kingdom,0,0,24
3,514509,15056P,EDWARDIAN PARASOL PINK,72,2010-07-04 13:10:00,4.95,15369,United Kingdom,0,0,72
4,492444,47566,PARTY BUNTING,6,2009-12-17 11:06:00,4.65,18097,United Kingdom,0,0,6


## 3.3 Outlier Detection:

In [19]:
numerical_cols = ['Quantity_abs', 'Price']   

In [20]:
for col in numerical_cols:
    Q1 = np.percentile(train[col], 25)
    Q3 = np.percentile(train[col], 75)
    IQR = Q3 - Q1
    multiplier = 1.5
    lower_bound = Q1 - multiplier * IQR
    upper_bound = Q3 + multiplier * IQR
    
    print(f"The lower bound for {col} is: {lower_bound}")
    print(f"The upper bound for {col} is: {upper_bound}")
    outliers_below_lower_bound = train[train[col] <= lower_bound]
    print(f"The number of transactions flagged as outliers below the lower bound ({lower_bound}) are: {len(train[train[col] <= lower_bound]):,}")
    print(f"The number of transactions flagged as outliers above the upper bound ({upper_bound}) are: {len(train[train[col]>= upper_bound]):,}")
    print(f"{'-' * 50}")
    

The lower bound for Quantity_abs is: -13.0
The upper bound for Quantity_abs is: 27.0
The number of transactions flagged as outliers below the lower bound (-13.0) are: 0
The number of transactions flagged as outliers above the upper bound (27.0) are: 26,064
--------------------------------------------------
The lower bound for Price is: -2.5
The upper bound for Price is: 7.5
The number of transactions flagged as outliers below the lower bound (-2.5) are: 0
The number of transactions flagged as outliers above the upper bound (7.5) are: 34,774
--------------------------------------------------


In [21]:
print(f"The max number of ordered quantity is: {train['Quantity'].max():,}")
print(f"The max number of returned quantity is: {train['Quantity'].min():,}")
print(f"The minimum number of returned quantity is: {train[train['Quantity'] < 0].max()['Quantity']:,}")

The max number of ordered quantity is: 74,215
The max number of returned quantity is: -74,215
The minimum number of returned quantity is: -1


In [25]:
print(f"The maximum price is: {train['Price'].max():,}")
print(f"The minimum price is: {train['Price'].min():,}")


The maximum price is: 25,111.09
The minimum price is: 0.001


## Capping/clipping the numerical data at 99th quantile:

In [58]:
numerical_cols = ['Quantity_abs', 'Price']   
caps = {
        'Quantity_abs': train['Quantity_abs'].quantile(0.99),
        'Price': train['Price'].quantile(0.99)
       }

In [59]:
for col, cap in caps.items():
    train[f'{col}-capped'] = train[col].clip(upper=cap)

In [60]:
train.head(2)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,orders_returned,orders_cancelled,Quantity_abs,Quantity_abs-capped,Price-capped
0,537472,22594,CHRISTMAS GINGHAM TREE,24,2010-12-07 11:11:00,0.85,12971,United Kingdom,0,0,24,24,0.85
1,532756,22984,"CARD, GINGHAM ROSE",12,2010-11-14 13:39:00,0.42,14769,United Kingdom,0,0,12,12,0.42


In [61]:
train.shape

(396164, 13)

In [62]:
train_copy_for_tree.shape

(396164, 11)